# Import Libraries

In [ ]:
import cv2
import numpy as np
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input

# Configuration

In [ ]:
model_path = "efficientnetb0_baseline.keras"

image_size = (160, 160)

class_dir = r"C:\Users\ghwns\HJ_git\CV-Projects\real-time-daily-activity-recognizer\images"
label_list = sorted(os.listdir(class_dir))

# Load Model

In [ ]:
model = load_model(model_path)
print("Model loaded successfully.")

# Initialize Webcam

In [ ]:
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    raise RuntimeError("❌ Failed to open webcam.")

# Real-Time Inference Loop

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess frame
    img = cv2.resize(frame, image_size)
    img_array = np.expand_dims(img.astype(np.float32), axis=0)
    img_preprocessed = preprocess_input(img_array)

    # Predict activity
    preds = model.predict(img_preprocessed, verbose=0)
    pred_class = np.argmax(preds[0])
    pred_label = label_list[pred_class]
    confidence = preds[0][pred_class]

    # Draw prediction on frame
    text = f"{pred_label} ({confidence * 100:.1f}%)"
    cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    # Show frame
    cv2.imshow("Real-Time Activity Recognition", frame)

    # Quit with 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release Resources

In [ ]:
cap.release()
cv2.destroyAllWindows()